# Chronos-2 (120M) — M6 Rounds 1–12 zero-shot inference

Third model of the comparison, after Chronos-T5 Base (200M) and Financial
Chronos-Small (46M). Same 12 Stage 3 contexts, same horizon, same asset
ordering — but Chronos-2 differs from the earlier two in **two fundamental
ways**, and this notebook is built around those differences.

| Item | Value |
|---|---|
| Model | `amazon/chronos-2` (≈120M parameters), **zero-shot** |
| Rounds | 1 … 12 |
| Context | 512 weekday daily log returns per asset, unchanged Stage 3 files |
| Horizon | 20 weekdays |
| Assets | 100 official M6 assets, official order |
| **Task shape** | **one multivariate task per round: `(1, 100, 512)`** |
| **Output** | **native quantiles, not sampled trajectories** |
| Saved array | `(100 assets, 20 days, 21 quantiles)` |

**1. Multivariate, not univariate.** The earlier Chronos models are univariate:
each asset was forecast on its own, in batches of 10 purely for speed. Chronos-2
takes all 100 return series **together as a single forecasting task** of shape
`(1 task, 100 variates, 512 observations)`, so the model can share information
across assets. All 100 are targets — none are covariates. There is no per-asset
loop and no batch size to tune here. `cross_learning` stays `False`: it shares
information *between separate tasks*, and we have exactly one task, which is
already jointly modelled.

**2. Quantiles, not samples.** Chronos-2 emits its native quantile levels
directly instead of sampled paths, so there is no `num_samples` and no random
sampling to seed. The levels are read from the loaded pipeline
(`pipeline.quantiles`) rather than hard-coded — the documented set is 21 levels:
0.01, 0.05, 0.10, 0.15, …, 0.90, 0.95, 0.99.

**No preprocessing here.** The Stage 3 contexts are passed through unchanged: no
recalculated returns, no external standardising or normalising, no clipping,
smoothing, backfilling or imputation. Chronos-2 does its own model-native scaling
and missing-value handling, so CARR's (round 1) and OGN's (all rounds) genuine
leading `NaN`s are handed to the model as-is. The only change made to the values
is a float64 → float32 dtype cast for the tensor.

**Scope.** Raw quantile forecasts only. No M6 post-processing of any kind: no
four-week returns, no ranking, no quintiles, no RPS, no DRE adjustment, and
**no conversion of quantiles into sampled trajectories**. That methodological
step is deliberately deferred — the quantile output is a different object from
the earlier models' samples and converting it deserves its own decision.

**Run the sections in order, top to bottom.** Each round is saved to Drive and
re-verified as soon as it finishes.

## 2. Install Chronos-2

`chronos-forecasting==2.3.1` pinned to match the version used for the earlier two
models, and Transformers held on a 4.x release — Transformers 5.x is not
compatible with this Chronos release.

Colab may ask you to restart the runtime after installing. If it does, restart
and then re-run from this cell.

In [2]:
%pip install -q "chronos-forecasting==2.3.1" "transformers>=4.41,<5"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.6/80.6 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 147.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


## 3. Imports, versions and settings

Prints the exact package versions for reproducibility, then defines the settings.
Note what is absent compared with the earlier notebooks: no `NUM_SAMPLES` and no
`SERIES_BATCH_SIZE`, because Chronos-2 returns quantiles and takes all 100 assets
as one task.

In [1]:
import hashlib
import traceback
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from chronos import Chronos2Pipeline

# --- Model ------------------------------------------------------------------
MODEL_ID = "amazon/chronos-2"
MODEL_SIZE = "~120M parameters"

# --- Experiment settings ----------------------------------------------------
CONTEXT_LENGTH = 512
PREDICTION_LENGTH = 20
RANDOM_SEED = 42          # set for reproducibility; quantile output is deterministic
ROUNDS = range(1, 13)

# Chronos-2 shares information across the variates of ONE task. cross_learning
# shares information across SEPARATE tasks - we have a single task, so it stays
# False (the API default).
CROSS_LEARNING = False

STRICT_CONTEXT_HASH = True

# Documented native quantile levels; the actual levels are read from the loaded
# pipeline in section 8 and these are used only to check them.
EXPECTED_QUANTILE_LEVELS = (
    [0.01, 0.05] + [round(0.05 * i, 2) for i in range(2, 19)] + [0.95, 0.99]
)

# --- Official M6 round schedule (Stage 3, pre-specified) --------------------
# round -> context start, origin (= context end), forecast start, forecast end
ROUND_SCHEDULE = {
    1:  ("2020-03-19", "2022-03-04", "2022-03-07", "2022-04-01"),
    2:  ("2020-04-16", "2022-04-01", "2022-04-04", "2022-04-29"),
    3:  ("2020-05-14", "2022-04-29", "2022-05-02", "2022-05-27"),
    4:  ("2020-06-11", "2022-05-27", "2022-05-30", "2022-06-24"),
    5:  ("2020-07-09", "2022-06-24", "2022-06-27", "2022-07-22"),
    6:  ("2020-08-06", "2022-07-22", "2022-07-25", "2022-08-19"),
    7:  ("2020-09-03", "2022-08-19", "2022-08-22", "2022-09-16"),
    8:  ("2020-10-01", "2022-09-16", "2022-09-19", "2022-10-14"),
    9:  ("2020-10-29", "2022-10-14", "2022-10-17", "2022-11-11"),
    10: ("2020-11-26", "2022-11-11", "2022-11-14", "2022-12-09"),
    11: ("2020-12-24", "2022-12-09", "2022-12-12", "2023-01-06"),
    12: ("2021-01-21", "2023-01-06", "2023-01-09", "2023-02-03"),
}

# SHA-256 of the repository's Stage 3 context files - the same 12 files used by
# both earlier models.
EXPECTED_CONTEXT_SHA256 = {
    1:  "ba19ad0af578e6ecb4e1d7f70fa509f2c0c24abda93e88902b7d5c1252b58d80",
    2:  "4b1854a641bb0229231d151301c8294c3f4cdf313dabfebff6d61e7a7d7e50fa",
    3:  "efef3a5bf1b1f4504761e6899d83cb26e62e097ad29e8363dcacc69c13b37798",
    4:  "541ce612db093729e3eee4be20f8e1a1aaa1a042cdd723cd119cd2f04effbe91",
    5:  "b9d4f19490cc9da3260ccce91739213a2936d3c0a238d6e29346a4d1a727421f",
    6:  "b3f3265fd0283e807033a330ff0679f0aaec0bea51aa3d7e67403a58cb3c2478",
    7:  "80846de8dacbd539012a01b0165a92a8681488e355bbffb8f2f06574827b6f4a",
    8:  "03e2441a200633b4ea0ed21184857ab8def4d8995feca82bdfafc2945e409e67",
    9:  "b9326f356a5feed2db3821125733c3ba2af4a988ad570be1e67fb8a590f4a4de",
    10: "9326663904ba2bc66a2f87975169d3e6c00371469a28f8cfd557ccc4aa9b19ce",
    11: "1c6ef423f44fc01e5df0e2c46dbe41b83bc24cd31c94d82a3371f2f91eeb611a",
    12: "d2ac16cd5c815a4a97e836986fd5e13098015af1885e31785dcd4349d40cbcdf",
}

# Genuine leading missing history, per Stage 3 (verified against the repo files).
EXPECTED_LEADING_NAN = {r: {"OGN": 302 - 20 * (r - 1)} for r in ROUNDS}
EXPECTED_LEADING_NAN[1] = {"CARR": 1, "OGN": 302}

# Official M6 asset order - used to verify the contexts, not to reorder them.
OFFICIAL_ASSET_ORDER = [
    "ABBV", "ACN", "AEP", "AIZ", "ALLE", "AMAT", "AMP", "AMZN", "AVB", "AVY",
    "AXP", "BDX", "BF-B", "BMY", "BR", "CARR", "CDW", "CE", "CHTR", "CNC",
    "CNP", "COP", "CTAS", "CZR", "DG", "DPZ", "DRE", "DXC", "EWA", "EWC",
    "EWG", "EWH", "EWJ", "EWL", "EWQ", "EWT", "EWU", "EWY", "EWZ", "FTV",
    "GOOG", "GPC", "GSG", "HIG", "HIGH.L", "HST", "HYG", "IAU", "ICLN",
    "IEAA.L", "IEF", "IEFM.L", "IEMG", "IEUS", "IEVL.L", "IGF", "INDA",
    "IUMO.L", "IUVL.L", "IVV", "IWM", "IXN", "JPEA.L", "JPM", "KR", "LQD",
    "MCHI", "META", "MVEU.L", "OGN", "PG", "PPL", "PRU", "PYPL", "RE",
    "REET", "ROL", "ROST", "SEGA.L", "SHY", "SLV", "SPMV.L", "TLT", "UNH",
    "URI", "V", "VRSK", "VXX", "WRK", "XLB", "XLC", "XLE", "XLF", "XLI",
    "XLK", "XLP", "XLU", "XLV", "XLY", "XOM",
]
N_ASSETS = len(OFFICIAL_ASSET_ORDER)

print("Versions")
print(f"  chronos-forecasting : {version('chronos-forecasting')}")
print(f"  transformers        : {version('transformers')}")
print(f"  torch               : {torch.__version__}")
print(f"  torch CUDA build    : {torch.version.cuda}")
print(f"  CUDA available      : {torch.cuda.is_available()}")
print(f"\nExperiment: {MODEL_ID} ({MODEL_SIZE}), zero-shot, rounds "
      f"{min(ROUNDS)}-{max(ROUNDS)}")
print(f"Model input per round : (1, {N_ASSETS}, {CONTEXT_LENGTH})  "
      "= 1 multivariate task x 100 target variates x 512 observations")
print(f"Saved array per round : ({N_ASSETS}, {PREDICTION_LENGTH}, "
      f"{len(EXPECTED_QUANTILE_LEVELS)})  = assets x forecast days x quantiles")

Versions
  chronos-forecasting : 2.3.1
  transformers        : 4.57.6
  torch               : 2.11.0+cu128
  torch CUDA build    : 12.8
  CUDA available      : True

Experiment: amazon/chronos-2 (~120M parameters), zero-shot, rounds 1-12
Model input per round : (1, 100, 512)  = 1 multivariate task x 100 target variates x 512 observations
Saved array per round : (100, 20, 21)  = assets x forecast days x quantiles


## 4. Mount Google Drive

Supplies the existing context files and stores the outputs permanently.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 5. Configure paths

`DRIVE_CONTEXT_DIR` is the folder both earlier models already read — the same 12
context CSVs, not another copy.

**`DRIVE_OUTPUT_DIR` is the one path to set before running.** It is created
automatically and nothing is written near the other models' outputs.

In [3]:
# Existing shared context folder - same files as the earlier model runs.
DRIVE_CONTEXT_DIR = Path("/content/drive/MyDrive/HonoursResearch/Round_1_Context")

# >>> SET THIS BEFORE RUNNING - the only path you need to change <<<
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/HonoursResearch/outputs/Chronos-2")

OUTPUT_PREFIX = "chronos_2"


def context_path(round_number: int) -> Path:
    return DRIVE_CONTEXT_DIR / f"round_{round_number:02d}_context.csv"


def quantiles_path(round_number: int) -> Path:
    return DRIVE_OUTPUT_DIR / f"{OUTPUT_PREFIX}_round{round_number:02d}_quantiles.npz"


if not DRIVE_CONTEXT_DIR.is_dir():
    raise FileNotFoundError(
        f"Context folder not found:\n  {DRIVE_CONTEXT_DIR}\n"
        "It must be the SAME folder used by the earlier model notebooks, "
        "containing round_01_context.csv .. round_12_context.csv."
    )

DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Contexts (existing, shared): {DRIVE_CONTEXT_DIR}")
print(f"Output folder (Chronos-2)  : {DRIVE_OUTPUT_DIR}")
print(f"Example output filename    : {quantiles_path(1).name}")

Contexts (existing, shared): /content/drive/MyDrive/HonoursResearch/Round_1_Context
Output folder (Chronos-2)  : /content/drive/MyDrive/HonoursResearch/outputs/Chronos-2
Example output filename    : chronos_2_round01_quantiles.npz


## 6. Check the 12 context files

Presence and byte-identity against the Stage 3 digests, before the model loads —
this is also what proves Chronos-2 sees exactly the inputs the other two models
saw, which is what makes the three-way comparison fair.

In [4]:
missing, mismatched = [], []
CONTEXT_SHA256 = {}

for r in ROUNDS:
    path = context_path(r)
    if not path.is_file():
        missing.append(path.name)
        continue
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    CONTEXT_SHA256[r] = digest
    status = "match" if digest == EXPECTED_CONTEXT_SHA256[r] else "DIFFERS"
    if status == "DIFFERS":
        mismatched.append(r)
    print(f"  round {r:02d}: {path.name}  sha256 {digest[:16]}...  ({status})")

if missing:
    raise FileNotFoundError(
        "Missing context files in "
        f"{DRIVE_CONTEXT_DIR}:\n  " + "\n  ".join(missing)
    )
if mismatched and STRICT_CONTEXT_HASH:
    raise ValueError(
        f"Contexts for round(s) {mismatched} are not byte-identical to the "
        "Stage 3 files. Re-copy them from Data/processed/rolling_origins/."
    )

print(f"\nAll {len(CONTEXT_SHA256)} context files present and verified "
      "(identical to those used by Chronos-T5 Base and Financial Chronos).")

  round 01: round_01_context.csv  sha256 ba19ad0af578e6ec...  (match)
  round 02: round_02_context.csv  sha256 4b1854a641bb0229...  (match)
  round 03: round_03_context.csv  sha256 efef3a5bf1b1f450...  (match)
  round 04: round_04_context.csv  sha256 541ce612db093729...  (match)
  round 05: round_05_context.csv  sha256 b9d4f19490cc9da3...  (match)
  round 06: round_06_context.csv  sha256 b3f3265fd0283e80...  (match)
  round 07: round_07_context.csv  sha256 80846de8dacbd539...  (match)
  round 08: round_08_context.csv  sha256 03e2441a200633b4...  (match)
  round 09: round_09_context.csv  sha256 b9326f356a5feed2...  (match)
  round 10: round_10_context.csv  sha256 9326663904ba2bc6...  (match)
  round 11: round_11_context.csv  sha256 1c6ef423f44fc01e...  (match)
  round 12: round_12_context.csv  sha256 d2ac16cd5c815a4a...  (match)

All 12 context files present and verified (identical to those used by Chronos-T5 Base and Financial Chronos).


## 7. GPU check

A GPU is required; the cell stops rather than silently running on CPU.

In [5]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected. Runtime > Change runtime type > Hardware "
        "accelerator: GPU, then re-run from section 3."
    )

GPU_NAME = torch.cuda.get_device_name(0)
print(f"GPU        : {GPU_NAME}")
print(f"CUDA (torch): {torch.version.cuda}")

GPU        : NVIDIA L4
CUDA (torch): 12.8


## 8. Load Chronos-2 — **once**

> ⚠️ **This downloads/loads `amazon/chronos-2` onto the GPU.** The weights go to
> the Colab runtime cache, never into the repository.

Loaded once and reused for all 12 rounds. Zero-shot inference only: no training,
fine-tuning or weight modification. The cell also reads the model's **native
quantile levels** from the pipeline and checks them against the documented 21
levels rather than imposing a hard-coded list.

In [6]:
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

pipeline = Chronos2Pipeline.from_pretrained(
    MODEL_ID,
    device_map="cuda",
)

# Native quantile levels, taken from the loaded model.
QUANTILE_LEVELS = np.asarray(pipeline.quantiles, dtype=np.float64)
N_QUANTILES = int(QUANTILE_LEVELS.size)

assert np.all(np.diff(QUANTILE_LEVELS) > 0), "Quantile levels are not increasing"
assert N_QUANTILES == len(EXPECTED_QUANTILE_LEVELS), (
    f"Expected {len(EXPECTED_QUANTILE_LEVELS)} native quantile levels, "
    f"the loaded model reports {N_QUANTILES}: {QUANTILE_LEVELS.tolist()}"
)
if not np.allclose(QUANTILE_LEVELS, EXPECTED_QUANTILE_LEVELS):
    print("NOTE: the model's quantile levels differ from the documented set; "
          "the model's own levels are used and saved.")

n_params = sum(p.numel() for p in pipeline.model.parameters())
MODEL_LOADED_ONCE = True

print(f"Loaded: {MODEL_ID}")
print(f"  parameters      : {n_params:,} ({n_params / 1e6:.1f}M)")
print(f"  device          : {GPU_NAME}")
print(f"  quantile levels : {N_QUANTILES} -> {QUANTILE_LEVELS.tolist()}")
print("  zero-shot inference only - no training or fine-tuning.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/478M [00:00<?, ?B/s]

Loaded: amazon/chronos-2
  parameters      : 119,477,664 (119.5M)
  device          : NVIDIA L4
  quantile levels : 21 -> [0.01, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 0.99]
  zero-shot inference only - no training or fine-tuning.


## 9. Rounds 1–12

### 9a. Helpers

* `load_and_validate_context` — identical to the earlier notebooks: 512 rows,
  `date` plus the 100 official assets in order, ascending unique dates, the
  round's official context start and origin, the correct 20-weekday forecast
  window, and the expected genuine leading `NaN`s with no interior gaps. It
  returns the `(100 assets, 512 steps)` matrix and proves it is a plain transpose
  of the loaded values.
* `forecast_round` — builds the `(1, 100, 512)` multivariate task, calls
  `pipeline.predict(...)`, and normalises the result to
  `(100 assets, 20 days, 21 quantiles)`. The API documents its return as
  `(n_variates, n_quantiles, prediction_length)`, but rather than trusting that,
  the function *inspects* the returned shape and transposes only if needed —
  20 days and 21 quantiles are different sizes, so the axes are unambiguous.
* `save_and_verify_round` — writes the NPZ, then reloads it and checks the four
  arrays, the shape, asset ordering, forecast dates, quantile levels, finiteness
  and quantile monotonicity.

In [7]:
def load_and_validate_context(round_number: int):
    """Load one Stage 3 context, validate it, and build the model input."""
    ctx_start, origin, fc_start, fc_end = ROUND_SCHEDULE[round_number]
    df = pd.read_csv(context_path(round_number), parse_dates=["date"])

    assert df.shape[0] == CONTEXT_LENGTH, (
        f"Round {round_number}: expected {CONTEXT_LENGTH} rows, found {df.shape[0]}"
    )
    assert list(df.columns) == ["date"] + OFFICIAL_ASSET_ORDER, (
        f"Round {round_number}: columns are not 'date' + the official M6 asset order"
    )
    dates = df["date"]
    assert dates.is_monotonic_increasing, f"Round {round_number}: dates not ascending"
    assert not dates.duplicated().any(), f"Round {round_number}: duplicate dates"
    assert dates.iloc[0] == pd.Timestamp(ctx_start), f"Round {round_number}: context start"
    assert dates.iloc[-1] == pd.Timestamp(origin), f"Round {round_number}: context end/origin"
    assert (dates <= pd.Timestamp(origin)).all(), (
        f"Round {round_number}: context contains a date after the origin"
    )

    matrix = df[OFFICIAL_ASSET_ORDER].to_numpy(dtype=np.float64).T
    assert matrix.shape == (N_ASSETS, CONTEXT_LENGTH)
    assert np.array_equal(
        matrix, df[OFFICIAL_ASSET_ORDER].to_numpy().T, equal_nan=True
    ), f"Round {round_number}: model input differs from the loaded context values"

    leading_missing, interior_missing = {}, {}
    for i, symbol in enumerate(OFFICIAL_ASSET_ORDER):
        row = matrix[i]
        n_lead = int(np.argmax(~np.isnan(row))) if np.isnan(row[0]) else 0
        if n_lead:
            leading_missing[symbol] = n_lead
        if np.isnan(row[n_lead:]).any():
            interior_missing[symbol] = int(np.isnan(row[n_lead:]).sum())
    assert not interior_missing, (
        f"Round {round_number}: unexpected interior NaNs {interior_missing}"
    )
    assert leading_missing == EXPECTED_LEADING_NAN[round_number], (
        f"Round {round_number}: leading NaN pattern changed - "
        f"expected {EXPECTED_LEADING_NAN[round_number]}, found {leading_missing}"
    )

    forecast_dates = pd.bdate_range(fc_start, fc_end)
    assert len(forecast_dates) == PREDICTION_LENGTH, f"Round {round_number}: forecast window"
    return matrix, df, forecast_dates


def forecast_round(context_matrix, round_number: int):
    """Forecast one round as a single multivariate task.

    Input  : (1, 100, 512) - one task, 100 target variates, 512 observations.
    Output : (100, 20, 21) - assets x forecast days x native quantiles.

    The only change made to the context values is the float32 cast the tensor
    needs; NaNs are passed through for Chronos-2's own missing-value handling.
    """
    model_input = context_matrix[np.newaxis, :, :].astype(np.float32)
    assert model_input.shape == (1, N_ASSETS, CONTEXT_LENGTH), (
        f"Round {round_number}: model input {model_input.shape}, "
        f"expected (1, {N_ASSETS}, {CONTEXT_LENGTH})"
    )

    outputs = pipeline.predict(
        inputs=model_input,
        prediction_length=PREDICTION_LENGTH,
        cross_learning=CROSS_LEARNING,
    )
    assert len(outputs) == 1, (
        f"Round {round_number}: expected 1 task in the output, got {len(outputs)}"
    )
    raw = outputs[0]
    raw = raw.to(torch.float32).cpu().numpy() if torch.is_tensor(raw) else np.asarray(raw)

    # Inspect the returned layout instead of assuming it. 20 days and 21
    # quantiles differ in size, so the axes can be identified unambiguously.
    if raw.shape == (N_ASSETS, N_QUANTILES, PREDICTION_LENGTH):
        quantile_forecasts = np.transpose(raw, (0, 2, 1))   # -> assets, days, quantiles
        layout = "(assets, quantiles, days) -> transposed"
    elif raw.shape == (N_ASSETS, PREDICTION_LENGTH, N_QUANTILES):
        quantile_forecasts = raw
        layout = "(assets, days, quantiles) -> already correct"
    else:
        raise ValueError(
            f"Round {round_number}: unrecognised output shape {raw.shape}; expected a "
            f"permutation of ({N_ASSETS}, {PREDICTION_LENGTH}, {N_QUANTILES})"
        )

    quantile_forecasts = np.ascontiguousarray(quantile_forecasts)
    assert quantile_forecasts.shape == (N_ASSETS, PREDICTION_LENGTH, N_QUANTILES)
    assert np.isfinite(quantile_forecasts).all(), (
        f"Round {round_number}: forecasts contain NaN or infinite values"
    )

    # Quantiles must not cross, for every asset and every forecast day.
    max_violation = float(-np.min(np.diff(quantile_forecasts, axis=2)))
    assert max_violation <= 1e-6, (
        f"Round {round_number}: quantiles are not monotonically ordered "
        f"(largest crossing {max_violation:.3e})"
    )

    return quantile_forecasts, layout, max_violation


def save_and_verify_round(round_number: int, quantile_forecasts, forecast_dates):
    """Save the raw quantile forecasts, then reload and verify the file."""
    path = quantiles_path(round_number)
    date_strings = np.array([d.strftime("%Y-%m-%d") for d in forecast_dates])

    np.savez_compressed(
        path,
        quantile_forecasts=quantile_forecasts,
        quantile_levels=QUANTILE_LEVELS,
        asset_symbols=np.array(OFFICIAL_ASSET_ORDER),
        forecast_dates=date_strings,
    )

    with np.load(path, allow_pickle=False) as reloaded:
        assert set(reloaded.files) == {
            "quantile_forecasts", "quantile_levels", "asset_symbols", "forecast_dates"
        }, f"Round {round_number}: unexpected arrays {sorted(reloaded.files)}"
        r_forecasts = reloaded["quantile_forecasts"]
        r_levels = reloaded["quantile_levels"]
        r_symbols = reloaded["asset_symbols"]
        r_dates = reloaded["forecast_dates"]

    assert r_forecasts.shape == (N_ASSETS, PREDICTION_LENGTH, N_QUANTILES), "reloaded shape"
    assert np.array_equal(r_forecasts, quantile_forecasts), "saved values differ"
    assert np.isfinite(r_forecasts).all(), "reloaded values not finite"
    assert np.allclose(r_levels, QUANTILE_LEVELS), "quantile levels changed on save/reload"
    assert list(r_symbols) == OFFICIAL_ASSET_ORDER, "asset ordering changed on save/reload"
    assert list(r_dates) == list(date_strings), "forecast dates changed on save/reload"
    return path


print("Helpers defined.")

Helpers defined.


### 9b. Run Rounds 1–12

One multivariate forecast per round, saved and re-verified immediately, so a
failure in a later round cannot lose earlier work. A failed round is recorded and
the loop continues.

If the GPU runs out of memory, note that the 100 assets **cannot** simply be
split into batches — that would break the single joint task this stage is
testing. Reduce memory some other way (a larger GPU) rather than by splitting.

In [8]:
records, failures = [], []
run_started_utc = datetime.now(timezone.utc)

for r in ROUNDS:
    _, origin, fc_start, fc_end = ROUND_SCHEDULE[r]
    print(f"\n=== Round {r:02d} | origin {origin} | forecast {fc_start} .. {fc_end} ===")
    try:
        context_matrix, context_df, forecast_dates = load_and_validate_context(r)
        print(f"  context validated: {context_matrix.shape} (assets x time steps) -> "
              f"task input (1, {N_ASSETS}, {CONTEXT_LENGTH})")

        started = datetime.now(timezone.utc)
        quantile_forecasts, layout, max_violation = forecast_round(context_matrix, r)
        elapsed = (datetime.now(timezone.utc) - started).total_seconds()

        saved_path = save_and_verify_round(r, quantile_forecasts, forecast_dates)
        records.append({
            "round": r, "origin": origin,
            "forecast_start": fc_start, "forecast_end": fc_end,
            "shape": tuple(quantile_forecasts.shape),
            "seconds": round(elapsed, 1), "file": saved_path.name,
            "context_sha256": CONTEXT_SHA256[r],
        })
        print(f"  output {tuple(quantile_forecasts.shape)} | api layout {layout}")
        print(f"  all finite, quantiles monotonic (max crossing {max_violation:.2e}) "
              f"| {elapsed:.1f}s")
        print(f"  saved and verified -> {saved_path.name}")

    except Exception as exc:
        failures.append((r, f"{type(exc).__name__}: {exc}"))
        print(f"  !! ROUND {r:02d} FAILED - earlier rounds remain saved")
        traceback.print_exc()

print(f"\nCompleted rounds: {[rec['round'] for rec in records]}")
if failures:
    print(f"FAILED rounds   : {[r for r, _ in failures]} (rerun these)")
else:
    print("No failures.")


=== Round 01 | origin 2022-03-04 | forecast 2022-03-07 .. 2022-04-01 ===
  context validated: (100, 512) (assets x time steps) -> task input (1, 100, 512)
  output (100, 20, 21) | api layout (assets, quantiles, days) -> transposed
  all finite, quantiles monotonic (max crossing -8.06e-05) | 0.9s
  saved and verified -> chronos_2_round01_quantiles.npz

=== Round 02 | origin 2022-04-01 | forecast 2022-04-04 .. 2022-04-29 ===
  context validated: (100, 512) (assets x time steps) -> task input (1, 100, 512)
  output (100, 20, 21) | api layout (assets, quantiles, days) -> transposed
  all finite, quantiles monotonic (max crossing -8.72e-05) | 0.1s
  saved and verified -> chronos_2_round02_quantiles.npz

=== Round 03 | origin 2022-04-29 | forecast 2022-05-02 .. 2022-05-27 ===
  context validated: (100, 512) (assets x time steps) -> task input (1, 100, 512)
  output (100, 20, 21) | api layout (assets, quantiles, days) -> transposed
  all finite, quantiles monotonic (max crossing -9.98e-05) |

## 10. Verify all 12 saved files

An independent pass over what is actually on Drive — nothing here relies on the
in-memory arrays.

In [9]:
ALL_VERIFIED = True
print(f"{'Rnd':>3}  {'file':<38} {'status':<7} {'shape':<18} {'order':<6} {'dates':<6} finite")

for r in ROUNDS:
    path = quantiles_path(r)
    if not path.is_file():
        ALL_VERIFIED = False
        print(f"{r:>3}  {path.name:<38} MISSING")
        continue

    _, _, fc_start, fc_end = ROUND_SCHEDULE[r]
    expected_dates = [d.strftime("%Y-%m-%d") for d in pd.bdate_range(fc_start, fc_end)]
    with np.load(path, allow_pickle=False) as f:
        keys = set(f.files)
        forecasts = f["quantile_forecasts"]
        levels = f["quantile_levels"]
        symbols = list(f["asset_symbols"])
        dates = list(f["forecast_dates"])

    ok_keys = keys == {"quantile_forecasts", "quantile_levels", "asset_symbols", "forecast_dates"}
    ok_shape = forecasts.shape == (N_ASSETS, PREDICTION_LENGTH, N_QUANTILES)
    ok_levels = levels.size == N_QUANTILES and np.allclose(levels, QUANTILE_LEVELS)
    ok_order = symbols == OFFICIAL_ASSET_ORDER
    ok_dates = dates == expected_dates
    ok_finite = bool(np.isfinite(forecasts).all())
    ok_monotonic = ok_shape and float(-np.min(np.diff(forecasts, axis=2))) <= 1e-6
    ok = all([ok_keys, ok_shape, ok_levels, ok_order, ok_dates, ok_finite, ok_monotonic])
    ALL_VERIFIED = ALL_VERIFIED and ok

    print(f"{r:>3}  {path.name:<38} {'OK' if ok else 'FAILED':<7} {str(forecasts.shape):<18} "
          f"{'yes' if ok_order else 'NO':<6} {'yes' if ok_dates else 'NO':<6} "
          f"{'yes' if ok_finite and ok_monotonic else 'NO'}")

print(f"\nAll 12 rounds verified on Drive: {ALL_VERIFIED}")

Rnd  file                                   status  shape              order  dates  finite
  1  chronos_2_round01_quantiles.npz        OK      (100, 20, 21)      yes    yes    yes
  2  chronos_2_round02_quantiles.npz        OK      (100, 20, 21)      yes    yes    yes
  3  chronos_2_round03_quantiles.npz        OK      (100, 20, 21)      yes    yes    yes
  4  chronos_2_round04_quantiles.npz        OK      (100, 20, 21)      yes    yes    yes
  5  chronos_2_round05_quantiles.npz        OK      (100, 20, 21)      yes    yes    yes
  6  chronos_2_round06_quantiles.npz        OK      (100, 20, 21)      yes    yes    yes
  7  chronos_2_round07_quantiles.npz        OK      (100, 20, 21)      yes    yes    yes
  8  chronos_2_round08_quantiles.npz        OK      (100, 20, 21)      yes    yes    yes
  9  chronos_2_round09_quantiles.npz        OK      (100, 20, 21)      yes    yes    yes
 10  chronos_2_round10_quantiles.npz        OK      (100, 20, 21)      yes    yes    yes
 11  chronos_2_rou

## 11. Completion summary

What to copy back into the repository once the run finishes.

In [10]:
print("Chronos-2 (120M) - M6 Rounds 1-12, zero-shot")
print(f"  model            : {MODEL_ID} ({MODEL_SIZE}), loaded once, no fine-tuning")
print(f"  task per round   : (1, {N_ASSETS}, {CONTEXT_LENGTH}) - one multivariate task, "
      "all 100 assets as targets")
print(f"  cross_learning   : {CROSS_LEARNING} (single task; variates are already joint)")
print(f"  rounds completed : {len(records)}/{len(list(ROUNDS))} "
      f"-> {[rec['round'] for rec in records]}")
if failures:
    print(f"  rounds FAILED    : {[r for r, _ in failures]}")
print(f"  output per round : ({N_ASSETS}, {PREDICTION_LENGTH}, {N_QUANTILES}) "
      "= assets x forecast days x native quantiles")
print(f"  quantile levels  : {N_QUANTILES} native levels, saved with every file")
print(f"  saved to         : {DRIVE_OUTPUT_DIR}")
print(f"  all verified     : {ALL_VERIFIED}")
print("  post-processing  : none (no four-week returns, quintiles, RPS, DRE change,")
print("                     and no conversion of quantiles into sampled trajectories)")
print("\nCopy the 12 NPZ files into the repository at:")
print("  Results/Chronos_2_120M/round_outputs/")

Chronos-2 (120M) - M6 Rounds 1-12, zero-shot
  model            : amazon/chronos-2 (~120M parameters), loaded once, no fine-tuning
  task per round   : (1, 100, 512) - one multivariate task, all 100 assets as targets
  cross_learning   : False (single task; variates are already joint)
  rounds completed : 12/12 -> [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
  output per round : (100, 20, 21) = assets x forecast days x native quantiles
  quantile levels  : 21 native levels, saved with every file
  saved to         : /content/drive/MyDrive/HonoursResearch/outputs/Chronos-2
  all verified     : True
  post-processing  : none (no four-week returns, quintiles, RPS, DRE change,
                     and no conversion of quantiles into sampled trajectories)

Copy the 12 NPZ files into the repository at:
  Results/Chronos_2_120M/round_outputs/
